In [ ]:
def first_order_similarity(arr):
    """
    Compute upper-triangle Spearman correlations between all pairs of timepoints.

    Parameters
    ----------
    arr : ndarray, shape (T, features)
        Time-by-feature matrix.

    Returns
    -------
    sim_vector : ndarray, shape (T*(T-1)/2,)
        Vector of pairwise Spearman correlations.
    """

    # Step 1: Rank transform each row (Spearman = Pearson on ranks)
    ranked = np.apply_along_axis(rankdata, 1, arr)

    # Step 2: Compute correlation matrix (vectorized)
    sim_matrix = np.corrcoef(ranked)

    # Step 3: Handle constant rows (std ≈ 0 → invalid correlations)
    stds = np.std(ranked, axis=1)
    invalid = np.isclose(stds, 0)

    if np.any(invalid):
        sim_matrix[invalid, :] = np.nan
        sim_matrix[:, invalid] = np.nan

    # Step 4: Extract upper triangle (excluding diagonal)
    iu = np.triu_indices_from(sim_matrix, k=1)
    sim_vector = sim_matrix[iu]

    return sim_vector

def safe_corr(a, b):
    """
    Compute Spearman correlation while ignoring NaNs.
    
    Returns nan if less than 2 valid points remain.
    """
    mask = ~np.isnan(a) & ~np.isnan(b)
    if mask.sum() < 2:
        return np.nan
    return spearmanr(a[mask], b[mask]).correlation


def compute_per_timepoint_rsa(fmri_corr_vec, model_corr_vec, T):
    """
    Compute second-order RSA per timepoint.
    
    fmri_corr_vec and model_corr_vec are flattened upper-triangle vectors
    of shape (T*(T-1)/2,)
    
    Returns
    -------
    timepoint_corrs : array, shape (T,)
        Spearman correlation for each timepoint. NaN if not enough valid pairs.
    """
    timepoint_corrs = np.zeros(T)
    
    # Precompute the indices of upper-triangle for each timepoint
    pair_idx = np.triu_indices(T, k=1)
    
    for t in range(T):
        # Select pairs involving timepoint t
        mask = (pair_idx[0] == t) | (pair_idx[1] == t)
        
        # Compute correlation safely
        timepoint_corrs[t] = safe_corr(fmri_corr_vec[mask], model_corr_vec[mask])
    
    return timepoint_corrs


 # Compute first-order similarity
fmri_corr_vec = first_order_similarity(fmri_list[i])
model_corr_vec = first_order_similarity(model_list[i])

T = fmri_list[i].shape[0]
timepoint_corrs = compute_per_timepoint_rsa(fmri_corr_vec, model_corr_vec, T)